# Fine-tune ruRoBERTa-large для детекции AI-текстов

Обучение трансформера на topic-matched данных (train_tm / val_tm).
Evaluation на test_tm, holdout (unseen Qwen3-32B) и adversarial.

In [ ]:
import json
import logging
import os
import time
import warnings
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings("ignore", message="Was asked to gather along dimension 0")
warnings.filterwarnings("ignore", module="sklearn.metrics")
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

## Конфигурация

In [ ]:
MODEL_NAME = "ai-forever/ruRoBERTa-large"
MAX_LENGTH = 512
EPOCHS = 5
BATCH_SIZE = 4
GRAD_ACCUM = 4
LR = 2e-5
EVAL_STEPS = 200
PATIENCE = 2
SEED = 42

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)

## Определение окружения

In [3]:
DATA_DIR = Path("/kaggle/input/datasets/vlasikq/ai-text-trust-splits-tm")
OUTPUT_DIR = Path("/kaggle/working/transformer")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model:      {MODEL_NAME}")
print(f"Data:       {DATA_DIR}")
print(f"Eff. batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"CUDA:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        mem_gb = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({mem_gb:.1f} GB)")

assert (DATA_DIR / "train_tm.jsonl").exists(), f"Данные не найдены: {DATA_DIR}"
print("Данные найдены")

Model:      ai-forever/ruRoBERTa-large
Data:       /kaggle/input/datasets/vlasikq/ai-text-trust-splits-tm
Eff. batch: 16
CUDA:       True
  GPU 0: Tesla T4 (14.6 GB)
  GPU 1: Tesla T4 (14.6 GB)
Данные найдены


## Dataset и метрики

In [4]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.encodings = tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=max_length, return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1)[:, 1].numpy()
    return {
        "f1_macro": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
        "auc_roc": roc_auc_score(labels, probs),
    }

## Загрузка данных

In [5]:
train_df = pd.read_json(DATA_DIR / "train_tm.jsonl", lines=True)
val_df = pd.read_json(DATA_DIR / "val_tm.jsonl", lines=True)
test_df = pd.read_json(DATA_DIR / "test_tm.jsonl", lines=True)
holdout_df = pd.read_json(DATA_DIR / "holdout_tm_unseen.jsonl", lines=True)

test_clean = test_df[test_df["source_model"] != "adversarial"].copy()
test_adv = test_df[test_df["source_model"] == "adversarial"].copy()

for name, df in [("train", train_df), ("val", val_df),
                 ("test_clean", test_clean), ("test_adv", test_adv),
                 ("holdout", holdout_df)]:
    vc = df["label"].value_counts()
    print(f"{name:12s}: {len(df):>6,}  (human={vc.get(0, 0):,}, AI={vc.get(1, 0):,})")

train       :  9,000  (human=4,500, AI=4,500)
val         :  1,200  (human=600, AI=600)
test_clean  :  1,800  (human=900, AI=900)
test_adv    :    511  (human=0, AI=511)
holdout     : 12,000  (human=6,000, AI=6,000)


## Токенизация

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = TextDataset(
    train_df["text"].tolist(), train_df["label"].tolist(), tokenizer, MAX_LENGTH
)
val_dataset = TextDataset(
    val_df["text"].tolist(), val_df["label"].tolist(), tokenizer, MAX_LENGTH
)

print(f"Train tokens: {train_dataset.encodings['input_ids'].shape}")
print(f"Val tokens: {val_dataset.encodings['input_ids'].shape}")

2026-04-02 09:26:01,928 WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Train tokens: torch.Size([9000, 512])
Val tokens: torch.Size([1200, 512])


## Обучение

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, problem_type="single_label_classification",
)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Parameters: {n_params:.0f}M")

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

start = time.time()
train_result = trainer.train()
elapsed = time.time() - start
print(f"\nОбучение завершено: {elapsed / 60:.1f} мин")
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")

pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: ai-forever/ruRoBERTa-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters: 355M


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Обучение завершено: 117.0 мин
Best checkpoint: /kaggle/working/transformer/checkpoints/checkpoint-600


## Сохранение модели

In [8]:
save_dir = OUTPUT_DIR / "model"
trainer.save_model(str(save_dir))
tokenizer.save_pretrained(str(save_dir))

val_results = trainer.evaluate()
print(f"Val F1: {val_results['eval_f1_macro']:.4f}")
print(f"Val Acc: {val_results['eval_accuracy']:.4f}")
print(f"Val AUC: {val_results['eval_auc_roc']:.4f}")

# save metrics
metrics = {
    "model_name": MODEL_NAME,
    "n_params_M": round(n_params, 1),
    "max_length": MAX_LENGTH,
    "val_f1_macro": round(val_results["eval_f1_macro"], 4),
    "val_accuracy": round(val_results["eval_accuracy"], 4),
    "val_auc_roc": round(val_results["eval_auc_roc"], 4),
    "train_loss": round(train_result.metrics["train_loss"], 4),
    "epochs_actual": round(trainer.state.epoch, 2),
    "effective_batch": BATCH_SIZE * GRAD_ACCUM,
    "train_runtime_min": round(elapsed / 60, 1),
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
}
(OUTPUT_DIR / "train_metrics.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8",
)
print(f"\nMetrics saved to {OUTPUT_DIR / 'train_metrics.json'}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Val F1: 0.9992
Val Acc: 0.9992
Val AUC: 1.0000

Metrics saved to /kaggle/working/transformer/train_metrics.json


## Evaluation на test_tm

In [9]:
def evaluate_split(df, name, tokenizer, trainer):
    ds = TextDataset(df["text"].tolist(), df["label"].tolist(), tokenizer, MAX_LENGTH)
    output = trainer.predict(ds)
    preds = output.predictions.argmax(axis=-1)
    probs = torch.softmax(
        torch.tensor(output.predictions, dtype=torch.float32), dim=-1
    )[:, 1].numpy()
    y = output.label_ids

    print(f"\n{'=' * 50}")
    print(f"{name} (n={len(y):,})")
    print("=" * 50)
    print(classification_report(y, preds, target_names=["human", "AI"], digits=3))

    f1 = f1_score(y, preds, average="macro")
    acc = accuracy_score(y, preds)
    try:
        auc = roc_auc_score(y, probs)
    except ValueError:
        auc = float("nan")
    print(f"F1-macro: {f1:.3f}  Accuracy: {acc:.3f}  AUC: {auc:.3f}")

    if "domain" in df.columns:
        print("\nPer-domain:")
        for dom in sorted(df["domain"].unique()):
            mask = df["domain"].values == dom
            if mask.sum() < 2:
                continue
            f1_d = f1_score(y[mask], preds[mask], average="macro")
            print(f"  {dom:8s}: F1={f1_d:.3f}  n={mask.sum()}")

    if "source_model" in df.columns:
        ai_mask = y == 1
        models = df.loc[ai_mask.astype(bool), "source_model"].unique()
        if len(models) > 1:
            print("\nPer-model (AI recall):")
            for mdl in sorted(models):
                mask = (df["source_model"].values == mdl) & ai_mask
                recall = preds[mask].mean()
                print(f"  {mdl:28s}: recall={recall:.3f}  n={mask.sum()}")

    return {"f1": f1, "acc": acc, "auc": auc, "preds": preds, "probs": probs}


res_test = evaluate_split(test_clean, "test_tm (clean)", tokenizer, trainer)


test_tm (clean) (n=1,800)
              precision    recall  f1-score   support

       human      0.999     0.996     0.997       900
          AI      0.996     0.999     0.997       900

    accuracy                          0.997      1800
   macro avg      0.997     0.997     0.997      1800
weighted avg      0.997     0.997     0.997      1800

F1-macro: 0.997  Accuracy: 0.997  AUC: 1.000

Per-domain:
  essay   : F1=0.997  n=600
  news    : F1=1.000  n=600
  wiki    : F1=0.995  n=600

Per-model (AI recall):
  GigaChat                    : recall=1.000  n=314
  llama-3.3-70b-versatile     : recall=1.000  n=313
  yandexgpt-lite              : recall=0.996  n=273


## Holdout - unseen Qwen3-32B

In [10]:
res_hold = evaluate_split(holdout_df, "holdout_tm (Qwen3 unseen)", tokenizer, trainer)


holdout_tm (Qwen3 unseen) (n=12,000)
              precision    recall  f1-score   support

       human      0.720     0.997     0.836      6000
          AI      0.995     0.613     0.759      6000

    accuracy                          0.805     12000
   macro avg      0.858     0.805     0.797     12000
weighted avg      0.858     0.805     0.797     12000

F1-macro: 0.797  Accuracy: 0.805  AUC: 0.973

Per-domain:
  essay   : F1=0.692  n=3275
  news    : F1=0.749  n=5007
  wiki    : F1=0.921  n=3718


## Adversarial

In [11]:
if len(test_adv) > 0:
    res_adv = evaluate_split(test_adv, "adversarial", tokenizer, trainer)
    recall_adv = res_adv["preds"].mean()
    print(f"\nAdversarial recall: {recall_adv:.3f}  avg_prob: {res_adv['probs'].mean():.3f}")
else:
    recall_adv = None


adversarial (n=511)
              precision    recall  f1-score   support

       human      0.000     0.000     0.000         0
          AI      1.000     0.908     0.952       511

    accuracy                          0.908       511
   macro avg      0.500     0.454     0.476       511
weighted avg      1.000     0.908     0.952       511

F1-macro: 0.476  Accuracy: 0.908  AUC: nan

Per-domain:
  essay   : F1=0.485  n=141
  news    : F1=0.469  n=238
  wiki    : F1=0.478  n=132

Adversarial recall: 0.908  avg_prob: 0.908


In [12]:
summary = pd.DataFrame([
    {"split": "test_tm", "F1": res_test["f1"], "Acc": res_test["acc"],
     "AUC": res_test["auc"], "n": len(test_clean)},
    {"split": "holdout (Qwen3)", "F1": res_hold["f1"], "Acc": res_hold["acc"],
     "AUC": res_hold["auc"], "n": len(holdout_df)},
    {"split": "adversarial", "F1": None, "Acc": None, "AUC": None,
     "n": len(test_adv), "recall_AI": recall_adv},
])
print(summary.to_string(index=False, float_format="%.3f"))

full_results = {
    "test_tm": {"f1": res_test["f1"], "acc": res_test["acc"], "auc": res_test["auc"]},
    "holdout_tm": {"f1": res_hold["f1"], "acc": res_hold["acc"], "auc": res_hold["auc"]},
    "adversarial_recall": float(recall_adv) if recall_adv is not None else None,
}
(OUTPUT_DIR / "eval_results.json").write_text(
    json.dumps(full_results, indent=2), encoding="utf-8",
)
print(f"\nResults saved: {OUTPUT_DIR / 'eval_results.json'}")

          split    F1   Acc   AUC     n  recall_AI
        test_tm 0.997 0.997 1.000  1800        NaN
holdout (Qwen3) 0.797 0.805 0.973 12000        NaN
    adversarial   NaN   NaN   NaN   511      0.908

Results saved: /kaggle/working/transformer/eval_results.json


## Кэш predictions для robustness matrix

Сохраняем predictions трансформера в JSON (~2 MB) для использования в 06_robustness.ipynb без загрузки модели (1.4 GB) и без GPU.

In [13]:
cached = {
    "test_tm": {
        "probs": res_test["probs"].tolist(),
        "preds": res_test["preds"].tolist(),
        "labels": test_clean["label"].tolist(),
    },
    "holdout_tm": {
        "probs": res_hold["probs"].tolist(),
        "preds": res_hold["preds"].tolist(),
        "labels": holdout_df["label"].tolist(),
    },
    "adversarial": {
        "probs": res_adv["probs"].tolist(),
        "preds": res_adv["preds"].tolist(),
        "labels": test_adv["label"].tolist(),
    },
}

out_path = OUTPUT_DIR / "cached_predictions.json"
out_path.write_text(json.dumps(cached), encoding="utf-8")
size_mb = out_path.stat().st_size / 1024**2
print(f"Saved: {out_path} ({size_mb:.1f} MB)")
for name, d in cached.items():
    print(f"  {name}: n={len(d['labels'])}")

Saved: /kaggle/working/transformer/cached_predictions.json (0.4 MB)
  test_tm: n=1800
  holdout_tm: n=12000
  adversarial: n=511


## Cleanup

Удаляем чекпоинты (~8 GB), оставляем модель + метрики + кэш predictions.

In [ ]:
checkpoints_dir = OUTPUT_DIR / "checkpoints"
if checkpoints_dir.exists():
    shutil.rmtree(checkpoints_dir)
    print(f"Удалены чекпоинты: {checkpoints_dir}")

print("\nФайлы в Output:")
total_mb = 0
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        size_mb = p.stat().st_size / 1024**2
        total_mb += size_mb
        print(f"  {p.relative_to(OUTPUT_DIR)}  ({size_mb:.1f} MB)")
print(f"\nИтого: {total_mb:.0f} MB")

Удалены чекпоинты: /kaggle/working/transformer/checkpoints

Файлы в Output:
  cached_predictions.json  (0.4 MB)
  eval_results.json  (0.0 MB)
  model/config.json  (0.0 MB)
  model/model.safetensors  (1355.6 MB)
  model/tokenizer.json  (5.1 MB)
  model/tokenizer_config.json  (0.0 MB)
  model/training_args.bin  (0.0 MB)
  train_metrics.json  (0.0 MB)

Итого: 1361 MB


## Выводы

ruRoBERTa-large (355M параметров) на topic-matched данных:

**In-distribution (test_tm):** F1=0.996, на уровне TF-IDF (0.996). По доменам стабильно: essay 0.997, news 0.998, wiki 0.992. Все три генератора из train детектируются с recall >= 0.997.

**Holdout (unseen Qwen3-32B):** F1=0.755, существенно ниже TF-IDF (0.904). AI recall = 0.544 при precision 0.983 - модель пропускает почти половину текстов Qwen3, но когда предсказывает "AI", ошибается редко. Per-domain: wiki 0.883 > news 0.781 > essay 0.549. Трансформер переобучается на стилистические паттерны генераторов из train и плохо переносится на unseen генератор.

**Adversarial:** recall = 0.886, лучший результат среди всех методов (TF-IDF 0.795, стилометрия 0.546). Контекстная модель устойчивее к парафразам, чем n-граммные и агрегированные подходы.

**Сравнение с другими методами:**

| Метод | test_tm F1 | holdout F1 | adv recall |
|-------|-----------|-----------|------------|
| TF-IDF (C) | 0.996 | 0.904 | 0.795 |
| ruRoBERTa | 0.996 | 0.755 | 0.886 |
| Stylometric | 0.973 | 0.505 | 0.546 |
| Style-only (B) | 0.759 | 0.802 | 0.546 |

Ни один метод не доминирует по всем осям. TF-IDF лучше обобщается на unseen генератор, трансформер устойчивее к adversarial-атакам